# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haritharamadass/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)



## 1. Method choice and why

### Method choice

I will use **Logistic Regression** to estimate whether a content item will decline in the following month.

This method fits my lane because the target, `declined_next_month`, is binary (decline / no decline). Logistic Regression is also simple and interpretable, and it produces probabilities that can be used to rank content for review.

I will use only March information available at the decision point. Client and content IDs will not be used as model features. The main evaluation metric will be **Precision@20**, so the learned model can be compared fairly with the Week-4 ranked baseline.

I am starting with a simple model rather than adding complexity before it is justified by the results.

In [1]:
# ML-08 — Section 1
# Method setup

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

FEATURES = [
    "impressions_31d",
    "ctr_31d",
    "avg_position_31d",
    "active_gsc_days",
]

TARGET = "declined_next_month"
GROUP = "client_hash_id"

print("Model: Logistic Regression")
print("Features:", FEATURES)
print("Target:", TARGET)
print("Grouped split by:", GROUP)
print("Random seed:", RANDOM_STATE)

Model: Logistic Regression
Features: ['impressions_31d', 'ctr_31d', 'avg_position_31d', 'active_gsc_days']
Target: declined_next_month
Grouped split by: client_hash_id
Random seed: 42


## 2. Split design

### Split design

I use an **80/20 grouped train-test split by `client_hash_id`** with a fixed random seed of 42.

Grouping by client keeps all content from the same client on only one side of the split. This reduces the risk that client-specific patterns appear in both training and test data.

The model uses only March 2026 features available at the decision point. April information is used only to create the future outcome `declined_next_month`.

The Week-4 baseline and the learned model will be evaluated on the **same held-out test rows** using **Precision@20** as the main ranking metric. This keeps the comparison fair.

In [2]:
# ============================================================
# ML-08 — Section 2
# Build the same March -> April feature frame and create
# an honest client-grouped train/test split.
# ============================================================

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face token loaded successfully.")


# ------------------------------------------------------------
# March = information available when the decision is made
# April = future month used only to create the outcome label
# ------------------------------------------------------------

MARCH = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

APRIL = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""


# ------------------------------------------------------------
# Rebuild the feature frame used for the Week-4 baseline
# ------------------------------------------------------------

feature_frame = con.sql(f"""
WITH march AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_31d,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_31d,

        SUM(
            gsc_avg_position * gsc_impressions
        ) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_impressions > 0
              AND gsc_avg_position > 0
        )
        /
        NULLIF(
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
                  AND gsc_impressions > 0
                  AND gsc_avg_position > 0
            ),
            0
        ) AS avg_position_31d,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS active_gsc_days

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (

    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_impressions,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_gsc_days

    FROM {APRIL}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.impressions_31d,
    m.clicks_31d,

    100.0 * m.clicks_31d
        / NULLIF(m.impressions_31d, 0) AS ctr_31d,

    m.avg_position_31d,
    m.active_gsc_days,

    (
        a.april_impressions
        < 0.80 * m.impressions_31d
    ) AS declined_next_month

FROM march AS m

INNER JOIN april AS a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id

WHERE
    m.impressions_31d >= 100
    AND m.active_gsc_days > 0
    AND a.april_gsc_days > 0
    AND m.impressions_31d IS NOT NULL
    AND m.avg_position_31d IS NOT NULL
    AND m.avg_position_31d > 0
    AND a.april_impressions IS NOT NULL
""").df()


print("\nFeature frame ready.")
print("Total rows:", len(feature_frame))
print("Clients:", feature_frame[GROUP].nunique())


# ------------------------------------------------------------
# Client-grouped 80/20 train-test split
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(
        feature_frame,
        feature_frame[TARGET],
        groups=feature_frame[GROUP]
    )
)

train_df = feature_frame.iloc[train_idx].copy()
test_df = feature_frame.iloc[test_idx].copy()


# ------------------------------------------------------------
# Verify that no client appears in both sets
# ------------------------------------------------------------

train_clients = set(train_df[GROUP])
test_clients = set(test_df[GROUP])

client_overlap = train_clients.intersection(test_clients)

print("\nSplit summary")
print("-" * 50)
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))

print("\nOutcome rates")
print("-" * 50)
print(
    "Training decline rate:",
    round(train_df[TARGET].mean(), 3)
)
print(
    "Test decline rate:",
    round(test_df[TARGET].mean(), 3)
)

assert len(client_overlap) == 0

print("\nGrouped split check passed: no client leakage.")

Hugging Face token loaded successfully.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Feature frame ready.
Total rows: 100893
Clients: 43

Split summary
--------------------------------------------------
Training rows: 85702
Test rows: 15191
Training clients: 34
Test clients: 9
Client overlap: 0

Outcome rates
--------------------------------------------------
Training decline rate: 0.533
Test decline rate: 0.412

Grouped split check passed: no client leakage.


## 3. Train + compare vs my baseline

### Training and baseline comparison

I train Logistic Regression on the grouped training set and rank the held-out test content by its predicted probability of declining next month.

For a fair comparison, I rebuild my Week-4 rule baseline on the **same held-out test rows**. Both methods are evaluated using the same future outcome and the same ranking metrics.

The main metric is **Precision@20**, because the practical question is which content items should be reviewed first. I also report Precision@50 to check whether the result is consistent beyond the first 20 items.

In [3]:
# ============================================================
# ML-08 — Section 3
# Train Logistic Regression and compare with Week-4 baseline
# on exactly the same held-out test rows.
# ============================================================

from sklearn.metrics import precision_score


# ------------------------------------------------------------
# 1. Train Logistic Regression
# ------------------------------------------------------------

X_train = train_df[FEATURES]
y_train = train_df[TARGET].astype(int)

X_test = test_df[FEATURES]
y_test = test_df[TARGET].astype(int)


model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "logistic_regression",
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    )
])

model.fit(X_train, y_train)


# Probability is used for ranking, not the hard 0/1 prediction.
test_results = test_df.copy()

test_results["model_probability"] = (
    model.predict_proba(X_test)[:, 1]
)


# ------------------------------------------------------------
# 2. Rebuild the Week-4 baseline rule on the SAME test rows
# ------------------------------------------------------------

baseline_test = test_df.copy()

baseline_test["ctr_opportunity"] = (
    (
        (baseline_test["avg_position_31d"] <= 3)
        & (baseline_test["ctr_31d"] < 0.15)
    )
    |
    (
        (baseline_test["avg_position_31d"] > 3)
        & (baseline_test["avg_position_31d"] <= 10)
        & (baseline_test["ctr_31d"] < 0.10)
    )
    |
    (
        (baseline_test["avg_position_31d"] > 10)
        & (baseline_test["avg_position_31d"] <= 20)
        & (baseline_test["ctr_31d"] < 0.05)
    )
)

baseline_test["baseline_score"] = (
    3 * baseline_test["ctr_opportunity"].astype(int)
    + 2 * (
        baseline_test["impressions_31d"] >= 1000
    ).astype(int)
    + 1 * (
        (baseline_test["impressions_31d"] >= 500)
        & (baseline_test["impressions_31d"] < 1000)
    ).astype(int)
    + 1 * (
        baseline_test["avg_position_31d"] <= 20
    ).astype(int)
)


# ------------------------------------------------------------
# 3. Helper: Precision@K
# ------------------------------------------------------------

def precision_at_k(df, score_column, k):
    ranked = (
        df
        .sort_values(
            by=[score_column, "impressions_31d"],
            ascending=[False, False]
        )
        .head(k)
    )

    return ranked[TARGET].astype(int).mean()


# ------------------------------------------------------------
# 4. Calculate the same metrics for baseline and model
# ------------------------------------------------------------

base_rate = y_test.mean()

baseline_p20 = precision_at_k(
    baseline_test,
    "baseline_score",
    20
)

baseline_p50 = precision_at_k(
    baseline_test,
    "baseline_score",
    50
)

model_p20 = precision_at_k(
    test_results,
    "model_probability",
    20
)

model_p50 = precision_at_k(
    test_results,
    "model_probability",
    50
)


comparison = pd.DataFrame({
    "Method": [
        "Held-out base rate",
        "Week-4 rule baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        base_rate,
        baseline_p20,
        model_p20
    ],
    "Precision@50": [
        base_rate,
        baseline_p50,
        model_p50
    ]
})

comparison["Precision@20"] = (
    comparison["Precision@20"].round(3)
)

comparison["Precision@50"] = (
    comparison["Precision@50"].round(3)
)


print("Model vs baseline — same test clients")
print("=" * 55)

display(comparison)


print("\nTest rows:", len(test_df))
print("Test clients:", test_df[GROUP].nunique())
print("Held-out base decline rate:", round(base_rate, 3))

Model vs baseline — same test clients


,Method,Precision@20,Precision@50
0,Held-out base rate,0.412,0.412
1,Week-4 rule baseline,0.700,0.500
2,Logistic Regression,0.800,0.720



Test rows: 15191
Test clients: 9
Held-out base decline rate: 0.412


## 4. Errors and interpretation

### Errors and interpretation

The Logistic Regression model performs better than the Week-4 rule baseline on the held-out clients, but it is not perfect.

At Precision@20, the model correctly identifies 16 of the top 20 ranked items as future declines, which means 4 highly ranked items are false positives. These are important because they show that a high predicted probability should be treated as decision support rather than certainty.

I inspect the model coefficients to understand which March features influence the ranking, and I review concrete false-positive and false-negative cases below. This helps check whether the model is learning plausible search-performance patterns rather than relying on suspicious or leaked information.

In [4]:
# ============================================================
# ML-08 — Section 4
# Errors and interpretation
# ============================================================

# ------------------------------------------------------------
# 1. Inspect standardized Logistic Regression coefficients
# ------------------------------------------------------------

logistic_model = model.named_steps["logistic_regression"]

coefficient_table = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": logistic_model.coef_[0]
})

coefficient_table["abs_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table = (
    coefficient_table
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

print("Standardized Logistic Regression coefficients")
print("=" * 55)

display(
    coefficient_table[
        ["feature", "coefficient"]
    ].round(3)
)


# ------------------------------------------------------------
# 2. Inspect errors in the model's top-20 ranked queue
# ------------------------------------------------------------

ranked_model = (
    test_results
    .sort_values(
        by=["model_probability", "impressions_31d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_model["rank"] = (
    ranked_model.index + 1
)

top20_model = ranked_model.head(20)

top20_false_positives = (
    top20_model[
        top20_model[TARGET].astype(int) == 0
    ]
    .copy()
)

print("\nTop-20 error summary")
print("=" * 55)
print("Top-20 items:", len(top20_model))
print(
    "Correct future declines:",
    int(top20_model[TARGET].astype(int).sum())
)
print(
    "False positives:",
    len(top20_false_positives)
)


# ------------------------------------------------------------
# 3. Show 3 concrete false-positive cases
# ------------------------------------------------------------

false_positive_examples = (
    top20_false_positives[
        [
            "rank",
            "content_hash_id",
            "impressions_31d",
            "ctr_31d",
            "avg_position_31d",
            "active_gsc_days",
            "model_probability",
            TARGET
        ]
    ]
    .head(3)
    .copy()
)

false_positive_examples["model_probability"] = (
    false_positive_examples["model_probability"].round(3)
)

false_positive_examples["ctr_31d"] = (
    false_positive_examples["ctr_31d"].round(3)
)

false_positive_examples["avg_position_31d"] = (
    false_positive_examples["avg_position_31d"].round(2)
)

print("\nThree highly ranked false-positive examples")
print("=" * 55)

display(false_positive_examples)


# ------------------------------------------------------------
# 4. Inspect false negatives using a 0.50 probability threshold
#    This is diagnostic only; Precision@K remains the main metric.
# ------------------------------------------------------------

diagnostic = test_results.copy()

diagnostic["predicted_class"] = (
    diagnostic["model_probability"] >= 0.50
).astype(int)

false_negatives = (
    diagnostic[
        (diagnostic[TARGET].astype(int) == 1)
        & (diagnostic["predicted_class"] == 0)
    ]
    .sort_values("model_probability")
)

print("\nDiagnostic threshold errors")
print("=" * 55)
print(
    "False negatives at 0.50 threshold:",
    len(false_negatives)
)

display(
    false_negatives[
        [
            "content_hash_id",
            "impressions_31d",
            "ctr_31d",
            "avg_position_31d",
            "active_gsc_days",
            "model_probability",
            TARGET
        ]
    ]
    .head(3)
    .round(3)
)

Standardized Logistic Regression coefficients


,feature,coefficient
0,ctr_31d,-0.442
1,active_gsc_days,0.430
2,impressions_31d,-0.156
3,avg_position_31d,-0.101



Top-20 error summary
Top-20 items: 20
Correct future declines: 16
False positives: 4

Three highly ranked false-positive examples


,rank,content_hash_id,impressions_31d,ctr_31d,avg_position_31d,active_gsc_days,model_probability,declined_next_month
0,1,content_e3a63930f7422987,447.0,0.0,2.03,31,0.673,False
9,10,content_e47b91d6e5daef37,143.0,0.0,4.61,31,0.670,False
14,15,content_11647416159eec65,499.0,0.0,3.73,31,0.670,False



Diagnostic threshold errors
False negatives at 0.50 threshold: 1829


,content_hash_id,impressions_31d,ctr_31d,avg_position_31d,active_gsc_days,model_probability,declined_next_month
92514,content_b96495c6b2911b12,160.0,7.500,33.694,28,0.000,True
50482,content_0a124186f945b8cd,259.0,7.336,18.444,31,0.001,True
41962,content_cdbf00ce4999800f,123.0,5.691,23.171,26,0.002,True
